# RAG audit demonstration

Connect FAISS retrieval to a structured NASA requirement audit. The saved outputs come from the local model run documented below. Set `RUN_MODEL = True` to rerun inference. Schema and citation validity do not establish verdict accuracy.

In Colab, choose a GPU runtime, then run all cells. Setup downloads the public repository and installs dependencies. Locally, open this notebook from the checkout after installing `requirements.txt`.

In [1]:
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/Aerospace-1C-compliance-copilot")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--branch", "alan/rag-audit-pipeline",
                        "https://github.com/Break-Through-Tech/Aerospace-1C-compliance-copilot.git",
                        str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "compliance_copilot").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository ready")

Repository ready


In [2]:
RUN_MODEL = False
REPORT = ROOT / "outputs/rag/demo.json"
if RUN_MODEL:
    REPORT = ROOT / "outputs/rag/notebook-run.json"
    REPORT.unlink(missing_ok=True)
    completed = subprocess.run([sys.executable, str(ROOT / "scripts/run_rag_demo.py"),
                                "--output", str(REPORT)], cwd=ROOT)
    if not REPORT.exists():
        raise RuntimeError("Model run failed before writing results; inspect the output above")
    if completed.returncode:
        print("The run contains audit errors; inspect them below.")
print("Displaying a new run" if RUN_MODEL else "Displaying the saved model run")

Displaying the saved model run


In [3]:
import json
import pandas as pd
from compliance_copilot.schemas import AuditResult

report = json.loads(REPORT.read_text())
results = [AuditResult.model_validate(row) for row in report["results"]]
print(json.dumps(report["config"], indent=2))
pd.DataFrame([{"requirement_id": r.requirement_id, "status": r.status,
               "verdict": r.audit.verdict if r.audit else None,
               "cited_swe_ids": ", ".join(c.swe_id for c in r.audit.citations) if r.audit else "",
               "error": r.error_code} for r in results])

{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "model_revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
  "top_k": 5,
  "do_sample": false,
  "max_new_tokens": 768,
  "device": "mps",
  "python": "3.12.13",
  "packages": {
    "torch": "2.14.1",
    "transformers": "4.57.6",
    "sentence-transformers": "5.7.0",
    "faiss-cpu": "1.15.1",
    "pydantic": "2.13.5"
  }
}


,requirement_id,status,verdict,cited_swe_ids,error
0,NASA-SR-001,error,None,,invalid_output
1,NASA-SR-002,error,None,,invalid_output
2,NASA-SR-003,ok,Gap,SWE-052,None
3,NASA-SR-004,ok,Meets,"SWE-050, SWE-053, SWE-042",None
4,NASA-SR-005,error,None,,invalid_output
5,NASA-SR-006,ok,Meets,SWE-055,None
6,NASA-SR-007,ok,Meets,SWE-057,None
7,NASA-SR-008,ok,Gap,"SWE-002, SWE-042, SWE-208, SWE-040, SWE-216",None
8,NASA-SR-009,ok,Meets,"SWE-135, SWE-201",None
9,NASA-SR-010,ok,Gap,SWE-062,None


In [4]:
for result in results:
    print(result.requirement_id)
    print(result.requirement_text)
    if result.audit:
        print(result.audit.verdict + ": " + result.audit.reasoning)
        for citation in result.audit.citations:
            clause = next(c for c in result.retrieved if c.swe_id == citation.swe_id)
            print(citation.swe_id, citation.section, repr(citation.quote))
            print(clause.source_url)
    else:
        print(result.error_code, result.error_message)
    print()

NASA-SR-001
The NASA Project Manager shall develop, maintain, and execute a Software Development Plan and a Software Security Plan covering the full software life cycle from requirements through retirement, addressing every applicable requirement of NPR 7150.2D per the approved Requirements Mapping Matrix.
invalid_output Model response failed the audit JSON schema

NASA-SR-002
The NASA software shall meet performance expectations as determined by the development team prior to delivery.
invalid_output Model response failed the audit JSON schema

NASA-SR-003
For Class B software, the NASA Project Manager shall perform, record, and maintain bidirectional traceability between higher-level and software requirements, software requirements and system hazards, software requirements and design components, design components and code, software requirements and software verifications, and software requirements and software non-conformances. The traceability records shall be updated within five bus

In [5]:
# Labels are joined only after inference, never supplied to the pipeline.
benchmark = pd.read_csv(ROOT / "data/benchmark/requirements.csv").set_index("requirement_id")
pd.DataFrame([{"requirement_id": r.requirement_id,
               "expected": benchmark.loc[r.requirement_id, "verdict"],
               "predicted": r.audit.verdict if r.audit else None,
               "expected_clause_retrieved": benchmark.loc[r.requirement_id, "swe_id"] in {c.swe_id for c in r.retrieved}}
              for r in results])

,requirement_id,expected,predicted,expected_clause_retrieved
0,NASA-SR-001,Meets,None,False
1,NASA-SR-002,Gap,None,False
2,NASA-SR-003,Meets,Gap,True
3,NASA-SR-004,Meets,Meets,True
4,NASA-SR-005,Partial,None,True
5,NASA-SR-006,Meets,Meets,True
6,NASA-SR-007,Meets,Meets,True
7,NASA-SR-008,Gap,Gap,False
8,NASA-SR-009,Meets,Meets,True
9,NASA-SR-010,Meets,Gap,True


## Interpretation

This small run demonstrates retrieval, generation, and validation. Expected verdicts are shown for inspection, not as a performance estimate. Review incorrect clause selection and incomplete source text separately from schema failures. The full 50-case evaluation belongs to Task 3.

See `docs/rag-pipeline.md` for the reusable callable interface, errors, source limitations, and reproduction settings.